# Natural Language Processing

## Importing the dataset with Kaggle

In [1]:
# import kagglehub
# path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
# print("Path to dataset files:", path)

## Importing the libraries

In [2]:
import numpy as np
import tensorflow as tf
import pandas as pd

## Importing the dataset with Pandas

In [3]:
dataset = pd.read_csv("IMDB_Dataset.csv")
x = dataset.iloc[:, : -1].values
y = dataset.iloc[:, -1].values

## Preprocessing the text

In [4]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
nltk.download('stopwords')

corpus = []
for i in range(len(dataset)):
  review = re.sub(r'<br\s*/?>', ' ', dataset['review'][i])
  review = re.sub(r'\s+', ' ', review).strip()
  review = review.split()
  ps = PorterStemmer()
  all_stopwords = stopwords.words('english')
  all_stopwords.remove('not')
  review = [ps.stem(word) for word in review if not word in set(all_stopwords)]
  review = ' '.join(review)
  corpus.append(review)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Splitting the dataset into training set and test set

In [5]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(corpus, y, test_size = 0.5, random_state = 0)

In [6]:
print(len(x))
print(len(x_train))
print(len(x_test))

50000
25000
25000


## Label encoding dependent variable y

In [7]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

## Vectorizing text into integers

In [8]:
from tensorflow.keras.layers import TextVectorization

vectorizer = TextVectorization (
    max_tokens = 10000,
    standardize = 'lower_and_strip_punctuation',
    split = 'whitespace',
    output_mode = 'int'
)

vectorizer.adapt(x_train)
vocab = vectorizer.get_vocabulary()

x_train_np = np.array(x_train, dtype=object)
x_test_np = np.array(x_test, dtype=object)

x_train_encoded = vectorizer(x_train_np)
x_test_encoded = vectorizer(x_test_np)

print(vocab)
print(len(vocab))

['', '[UNK]', np.str_('i'), np.str_('the'), np.str_('film'), np.str_('movi'), np.str_('it'), np.str_('not'), np.str_('one'), np.str_('like'), np.str_('thi'), np.str_('good'), np.str_('make'), np.str_('time'), np.str_('get'), np.str_('see'), np.str_('watch'), np.str_('even'), np.str_('movie'), np.str_('would'), np.str_('realli'), np.str_('charact'), np.str_('well'), np.str_('show'), np.str_('much'), np.str_('there'), np.str_('look'), np.str_('stori'), np.str_('and'), np.str_('scene'), np.str_('also'), np.str_('great'), np.str_('bad'), np.str_('love'), np.str_('go'), np.str_('first'), np.str_('that'), np.str_('think'), np.str_('play'), np.str_('end'), np.str_('way'), np.str_('made'), np.str_('but'), np.str_('thing'), np.str_('could'), np.str_('peopl'), np.str_('in'), np.str_('a'), np.str_('know'), np.str_('say'), np.str_('seem'), np.str_('act'), np.str_('seen'), np.str_('mani'), np.str_('plot'), np.str_('two'), np.str_('want'), np.str_('never'), np.str_('take'), np.str_('work'), np.str_(

## Creating the model

In [9]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense

model = Sequential()

model.add(Embedding(
    input_dim = len(vocab),
    output_dim = 128,
))

model.add(GlobalAveragePooling1D())
model.add(Dense(units = 64, activation = 'relu'))
model.add(Dense(units = 1, activation = 'sigmoid'))

## Compiling the model

In [10]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

## Training the model

In [12]:
model.fit(
    x_train_encoded,
    y_train,
    batch_size = 128,
    epochs = 10
)

Epoch 1/10
196/196 ━━━━━━━━━━━━━━━━━━━━ 51s 246ms/step - accuracy: 0.5109 - loss: 0.6945
Epoch 2/10
196/196 ━━━━━━━━━━━━━━━━━━━━ 81s 244ms/step - accuracy: 0.5351 - loss: 0.6910
Epoch 3/10
196/196 ━━━━━━━━━━━━━━━━━━━━ 48s 244ms/step - accuracy: 0.5465 - loss: 0.6871
Epoch 4/10
196/196 ━━━━━━━━━━━━━━━━━━━━ 84s 257ms/step - accuracy: 0.5848 - loss: 0.6756
Epoch 5/10
196/196 ━━━━━━━━━━━━━━━━━━━━ 46s 236ms/step - accuracy: 0.6262 - loss: 0.6426
Epoch 6/10
196/196 ━━━━━━━━━━━━━━━━━━━━ 83s 240ms/step - accuracy: 0.6700 - loss: 0.6030
Epoch 7/10
196/196 ━━━━━━━━━━━━━━━━━━━━ 56s 286ms/step - accuracy: 0.7049 - loss: 0.5809
Epoch 8/10
196/196 ━━━━━━━━━━━━━━━━━━━━ 74s 247ms/step - accuracy: 0.7463 - loss: 0.5363
Epoch 9/10
196/196 ━━━━━━━━━━━━━━━━━━━━ 83s 252ms/step - accuracy: 0.7588 - loss: 0.4896
Epoch 10/10
196/196 ━━━━━━━━━━━━━━━━━━━━ 49s 248ms/step - accuracy: 0.7537 - loss: 0.5059


## Predicting test set result

In [15]:
y_preds = model.predict(x_test_encoded)

782/782 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step


In [19]:
predictions = []

for pred in y_preds:
  if pred > 0.5:
    predictions.append(1)
  else:
    predictions.append(0)

## Making the confusion matrix

In [22]:
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, predictions)
print(cm)
accuracy_score(y_test, predictions)

[[ 9939  2622]
 [ 1383 11056]]


0.8398

## Preprocessing and Predicting a single review

In [42]:
pred_str = "This movie was so damn good!"

review_str = re.sub(r'<br\s*/?>', ' ', pred_str)
review_str = re.sub(r'\s+', ' ', review_str).strip()
review_str = review_str.split()
review_str = [ps.stem(word) for word in review_str if not word in set(all_stopwords)]
review_str = ' '.join(review_str)


np_pred_str = np.array(review_str, dtype = object)
np_pred_str = np_pred_str.reshape(1, -1)
pred_encoded_str = vectorizer(np_pred_str)
pred = model.predict(pred_encoded_str)

result = []

if pred > 0.5:
  print('The Review is Positive')
else:
  print('The Review is Negative')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
The Review is Positive
